# Predicción de precios de vivienda en California

Proyecto de Machine Learning supervisado sobre dos dominios distintos: predicción del precio medio de vivienda en California y predicción de abandono de clientes (churn) en banca. El desafío plantea dos preguntas centrales: *¿cuánto vale una vivienda dado su contexto?* y *¿qué clientes están en riesgo real de abandonar el banco?*

Este notebook recorre el pipeline completo — desde carga de datos y análisis exploratorio hasta feature engineering, entrenamiento, evaluación y diagnóstico de errores — para responder ambas preguntas con el estricto Ritual de los 7 pasos y una redacción honesta de los hallazgos: qué funciona, qué limita el modelo y cómo el propio RMSE nos mantiene humildes.


## Carga de datos

Trabajamos con dos datasets reales en formato CSV ubicados en `data/raw/`:

- **California Housing Prices**: precios de vivienda en California (1990) a nivel de bloque censal, con el target `median_house_value` en **dólares reales**.
- **Bank Customer Churn**: 10.000 clientes de un banco europeo con su indicador de abandono (`Exited`).

Ambos se cargan tal cual, sin modificaciones sobre los archivos originales.


In [ ]:
#  Imports y Configuración 

# Importamos las librerías de análisis y visualización de datos.
import pandas as pd          
import numpy as np           
import matplotlib.pyplot as plt  
import seaborn as sns        

# 1. División de Datos y Optimización (sklearn.model_selection) 
from sklearn.model_selection import train_test_split # train_test_split: division de datos en train y test (el modelo aprende con train y se mide con test, como en el reto).

# 2. Limpieza y Transformación de Datos  (sklearn.preprocessing e impute) 
from sklearn.preprocessing import StandardScaler, OneHotEncoder
#   StandardScaler: escala los números para que tengan media 0 y desvía estándar 1.
#     Crucial para Regresión Logística o K-Means: evita que una variable con números
#     gigantes (salario) opaque a una con números pequeños (edad).
#   OneHotEncoder: convierte texto/categorías (“Rojo”, “Verde”) en columnas de 0/1.
#     Los modelos sólo entienden números, así que este paso es obligatorio.
from sklearn.compose import ColumnTransformer
#   ColumnTransformer: aplica transformaciones DIFERENTES a columnas DIFERENTES en un
#     solo paso (ej: StandardScaler a numéricas + OneHotEncoder a texto a la vez).
from sklearn.pipeline import Pipeline
#   Pipeline: “fábrica” o cadena de montaje. Pega preprocesamiento + modelo en un solo
#     objeto; con .fit() limpia, transforma y entrena automáticamente y en orden.
from sklearn.impute import SimpleImputer
#   SimpleImputer: llena los datos faltantes (nulos/NaN) con el promedio, la mediana
#     o el valor más repetido de la columna (usamos mediana para total_bedrooms).
from sklearn.cluster import KMeans
#   KMeans: aprendizaje NO supervisado. Agrupa datos por similitud sin etiquetas
#     previas (creamos ZONAS GEOGRÁFICAS a partir de latitud/longitud).


# 3. Modelos de Machine Learning (Algoritmos)
from sklearn.linear_model import LinearRegression, LogisticRegression
#   LinearRegression: regresión clásica. Predice un valor continuo (precio de una casa)
#     dibujando la línea que mejor se ajusta (mínimos cuadrados / OLS).
#   LogisticRegression: para CLASIFICACIÓN binaria (predecir si ocurre un evento o no,
#     como el Churn o el Spam). Devuelve la probabilidad de pertenecer a la clase 1.


# 4. Metricas de Evaluacion de Modelos (sklearn.metrics)
from sklearn.metrics import (mean_squared_error, r2_score, accuracy_score,
                             precision_score, recall_score, classification_report,
                             confusion_matrix)
from sklearn.metrics import ConfusionMatrixDisplay
#   --- Para REGRESIÓN (predicción de números) ---
#   mean_squared_error: calcula el MSE (error cuadrático medio). Con raíz cuadrada da
#     el RMSE, el error promedio en las unidades originales ($).
#   r2_score (R²): qué porcentaje de la variación de los datos explica el modelo
#     (0 a 1, donde 1 = ajuste perfecto). Lo usamos para California.
#   --- Para CLASIFICACIÓN (predicción de categorías) ---
#   accuracy_score: % total de predicciones correctas (a cuántos acerté en total).
#   precision_score: de los clasificados como “Positivos”, cuántos lo eran realmente
#     (de los que predije que harían Churn, cuántos sí lo hicieron). Evita falsos positivos.
#   recall_score: sensibilidad. De los que realmente hicieron Churn, cuántos detectó.
#     Evita falsos negativos.
#   classification_report: resumen completo (accuracy, precision, recall, f1 por clase).
#   confusion_matrix: cruza valores reales vs predicciones (aciertos, falsos pos./neg.).
#   ConfusionMatrixDisplay: grafica la matriz de confusión con colores para leerla fácil.


# c) Configuracion general
import warnings
warnings.filterwarnings('ignore')          
sns.set_theme(style='whitegrid')          

SEED = 42                                  # semilla fija: misma partición en cada ejecución (reproducibilidad)

# Rutas relativas al notebook (data/raw/) y carga de ambos datasets
# Churn: dataset clásico de 10.000 clientes (abbas829/bank-customer-churn -> Bank_Churn.csv).
CAL = r'data/raw/housing.csv'              # dataset de regresión (California)
BNK = r'data/raw/Bank_Churn.csv'           # dataset de clasificación (Bank churn)

df_cal_raw = pd.read_csv(CAL)             
df_bnk_raw = pd.read_csv(BNK)              

print('California Housing:', df_cal_raw.shape[0], 'filas x', df_cal_raw.shape[1], 'columnas')
print('Bank Churn      :', df_bnk_raw.shape[0], 'filas x', df_bnk_raw.shape[1], 'columnas')


## Contexto: hallazgos clave del EDA

Antes de modelar, miremos los datos a la cara. Las decisiones de preparación se basan en lo que el análisis exploratorio revela, no en suposiciones.

### California Housing

- `total_bedrooms` tiene **207 nulos** que deben imputarse.
- `median_income` es el predictor más fuerte del precio (correlación ~0.69 en la serie completa).
- `ocean_proximity` es categórica (`<1H OCEAN`, `INLAND`, `ISLAND`, `NEAR BAY`, `NEAR OCEAN`) y debe codificarse.
- El target tiene un **techo artificial de $500,001**: las viviendas de lujo fueron recortadas a ese valor, lo que distorsiona el error. Se eliminan en el feature engineering.
- `latitude` y `longitude` capturan ubicación pero como coordenadas crudas; las convertiremos en **zonas geográficas con K-Means** para hacerlas interpretables.
- `total_rooms` y `total_bedrooms` están correlacionadas entre sí (multicolinealidad).

### Bank Churn

- El dataset está **libre de nulos** (10.000 filas de clientes de un banco europeo).
- El target `Exited` está **desbalanceado** (79.63% se queda / 20.37% se va) — el baseline de Accuracy es ~79.63%, y el **Recall será clave** para detectar fugas.
- `Age` es el predictor más fuerte del churn: los clientes que se van tienen una edad media de **44.8 años** vs 37.4 de los que se quedan.
- `Geography` revela un patrón regional: **Alemania** concentra el 32% de las bajas, vs ~16% en Francia y España — `Geography_Germany` será un predictor relevante.
- `IsActiveMember` tiene un **efecto protector**: los miembros activos abandonan menos (promedio 0.64 en se queda vs 0.36 en se va).
- `NumOfProducts` tiene una relación **no lineal** con el churn (más de 2 productos se asocia a mayor fuga); se conserva numérico para que el modelo capture la señal dominante.


In [ ]:
# eda rapido: california housing

print('\n========== EDA RÁPIDO: CALIFORNIA HOUSING ==========')
print(f'Dimensiones: {df_cal_raw.shape}')

print(f'\nNulos por columna:\n{df_cal_raw.isnull().sum()}')

# los modelos solo entienden números -> esto se tendrá que codificar (one-hot).
print('\nValores únicos en ocean_proximity:')
print(df_cal_raw['ocean_proximity'].value_counts())


In [ ]:
# eda rapido: bank churn

import pandas as pd
from IPython.display import display

print('\n========== EDA RÁPIDO: BANK CHURN ==========')
print('Dimensiones:', df_bnk_raw.shape)

# Se normalizan los datos a proporciones (0 a 1)
print('\nDistribución del target Exited:')
print(df_bnk_raw['Exited'].value_counts(normalize=True).round(4).to_string())

# Bajas (tasa) y clientes por país en tabla
baja_pais = df_bnk_raw.groupby('Geography')['Exited'].agg(['count','mean']).rename(columns={'count':'Clientes','mean':'Tasa de bajas'}).round(4)
print('\n--- Distribucion y Bajas por país (lado a lado) ---')
print(baja_pais)

# Edad media y miembros activos según Exited
edad_active = df_bnk_raw.groupby('Exited').agg(Edad_media=('Age','mean'), Miembros_activos=('IsActiveMember','mean')).round(3)
edad_active.index = ['Se queda (0)','Se va (1)']
print('\n--- Edad y actividad según Exited (lado a lado) ---')
print(edad_active)


## Feature engineering

Con los datos explorados, el siguiente paso es transformarlos en algo que el modelo pueda aprovechar mejor. Creamos nuevas variables a partir de las originales — densidades con sentido económico, interacciones y zonas geográficas — para darle al modelo mejor materia prima sin agregar información externa.


In [ ]:
# feature engineering: california
# Tres ideas del desafío: quitar el techo artificial del target, derivar ratios con sentido económico y crear ZONAS GEOGRÁFICAS con K-Means.

# Trabajamos sobre una copia para no tocar el DataFrame crudo.
df_cal = df_cal_raw.copy()

# El Techo artificial distorsiona el rmse por eso hay que eliminarlo
print('Filas con target == 500001 (techo):', (df_cal['median_house_value'] >= 500001).sum()) 
df_cal = df_cal[df_cal['median_house_value'] < 500001].reset_index(drop=True) # reset_index(drop=True) renumeran las filas de 0 en adelante tras filtrar.
print('Filas tras eliminar el techo:', len(df_cal))

# Ratios
df_cal['rooms_per_household'] = df_cal['total_rooms'] / df_cal['households']
df_cal['population_per_room'] = df_cal['population'] / df_cal['total_rooms']

# concepto del challenge (interacción / cross feature): (crear una variable nueva combinando dos variables existentes para capturar un efecto sinérgico)
# 3) age_x_income: multiplicar antigüedad x ingreso captura un EFECTO COMBINADO que ninguna variable ve por separado (una casa vieja y rica no equivale a vieja + rica).
df_cal['age_x_income'] = df_cal['housing_median_age'] * df_cal['median_income']  # El valor numérico de esta nueva columna se dispara únicamente cuando ambos valores son altos al mismo tiempo.

# concepto del challenge (aprendizaje NO supervisado dentro de una misión supervisada):
# 4) Zonas geográficas con K-Means. En lugar de pasar lat/long crudas (poco interpretables), agrupamos las ubicaciones en 8 zonas y usamos la zona como variable categórica.
#    importante: no usamos el target para agrupar -> NO hay fuga de datos.
geo = df_cal[['latitude', 'longitude']].values   # matriz de coordenadas

# Instanciamos K-Means con 8 clusters fijos por la semilla (reproducibilidad).
# n_init=10 repite el algoritmo 10 veces y se queda con el mejor agrupamiento.
kmeans = KMeans(n_clusters=8, random_state=SEED, n_init=10) #n_clusters= cantidad de grupos, n_init = cuantas veces hara el proceso de agrupacion

df_cal['geo_zone'] = kmeans.fit_predict(geo) #Calcula las distancias y agrupa todos tus distritos en 8 regiones.

# get_dummies: convierte la zona en columnas 0/1. drop_first evita redundancia (k-1 dummies).
geo = pd.get_dummies(df_cal['geo_zone'], prefix='geo_zone').astype(int)

# Concatenamos las dummies geográficas al dataset.
df_cal = pd.concat([df_cal, geo], axis=1)

# Nota: los nulos de total_bedrooms NO se imputan aquí a propósito;
#       los imputaremos dentro del Pipeline (paso siguiente) para no filtrar datos.
print('\nRatios derivados (muestra):')
print(df_cal[['rooms_per_household', 'population_per_room', 'age_x_income']].head(3))
print('\nZonas geográficas creadas:', list(geo.columns)[:5], '... total', len(geo.columns))


In [ ]:
# feature engineering: bank churn (limpieza y encoding)
# concepto del challenge (Paso 3 del Ritual — preparación de datos):
#   Aquí limpiamos el dataset de churn: descartamos identificadores inútiles y
#   convertimos las columnas de texto a números (porque el modelo solo entiende números).
df_bnk = df_bnk_raw.copy()
# concepto: CustomerId y Surname son IDENTIFICADORES (no aportan información).
#   Si los dejáramos, el modelo memorizaría clientes (sobreajuste) en vez de aprender patrones.
#   (el clásico Churn_Modelling.csv trae también RowNumber; este Bank_Churn.csv no lo incluye)
df_model = df_bnk.drop(columns=['CustomerId', 'Surname'])
# Separamos el target: la columna que queremos predecir (1 = se fue, 0 = se quedó).
#   La guardamos aparte para que no entre a las características.
y = df_model['Exited']
# concepto del challenge (encoding / one-hot sin redundancia):
#   Geography (France/Spain/Germany) y Gender (Male/Female) son texto -> get_dummies
#   las convierte en columnas 0/1. drop_first elimina una columna de cada clase
#   (evita la multicolinealidad perfecta y ya queda representada por los ceros).
df_model = pd.get_dummies(df_model, columns=['Geography', 'Gender'], drop_first=True)
# get_dummies puede devolver columnas booleanas (True/False); las pasamos a entero 0/1.
for c in df_model.select_dtypes(include='bool').columns:
    df_model[c] = df_model[c].astype(int)
# concepto del challenge: NumOfProducts tiene relación NO LINEAL con el churn
#   (0,1,3,4 productos son valores “atípicos”). Por ahora lo dejamos numérico:
#   el modelo lineal capturará la señal dominante; no forzamos ninguna transformación.
print('Dimensiones Bank listas:', df_model.shape)
print('Nulos restantes:', int(df_model.isnull().sum().sum()))
print('Columnas finales:', df_model.columns.tolist())


## Selección de features

Separamos características (X) de objetivo (y) para ambos problemas, dejando listos los conjuntos que alimentarán los modelos.


In [ ]:
# celda 10 - selección de features: california
# concepto del challenge (Paso 4 del Ritual — Feature selection):
#   Separamos qué variables serán la entrada (X) y cuál será el objetivo (y)
#   para la regresión. importante: el target y_cal queda en DÓLARES REALES (sin techo).
# Lista de columnas NUMÉRICAS que alimentarán el modelo de regresión.
num_cols = ['longitude', 'latitude', 'housing_median_age', 'total_rooms',
            'total_bedrooms', 'population', 'households', 'median_income',
            'rooms_per_household', 'population_per_room', 'age_x_income']
# Columnas CATEGÓRICAS (texto) que el Pipeline codificará con one-hot (ocean_proximity).
cat_cols = ['ocean_proximity']
# Columnas de las ZONAS GEOGRÁFICAS que creamos con K-Means (empiezan con “geo_zone_”).
geo_cols = [c for c in df_cal.columns if c.startswith('geo_zone_')]
# X_cal = entrada del modelo; y_cal = lo que queremos predecir (precio de la casa).
X_cal = df_cal[num_cols + cat_cols + geo_cols].copy()
y_cal = df_cal['median_house_value'].copy()
print('California X:', X_cal.shape, '| y:', y_cal.shape)
print('Columnas numéricas:', num_cols)
print('Columnas geográficas:', geo_cols)
print('Categórica:', cat_cols)
# NOTA: la selección de features de Bank Churn (X_bnk / y_bnk) se hace en
# la celda de División train/test, donde se define y_bnk desde 'Exited'.


## División train/test

Dividimos 80/20 **antes** de escalar. El test permanece “sellado” hasta la evaluación final. Para churn **estratificamos** por `y` para conservar la proporción de clases (73.46/26.54) en ambos conjuntos.


In [ ]:
# celda 12 - división train / test

# concepto clave del challenge (Paso 5 del Ritual — División):
#   Guardamos una parte de los datos que el modelo NUNCA verá durante el entrenamiento
#   (test). Ese conjunto “sellado” sirve para evaluar de forma honesta al final.
#   Dividir ANTES de escalar evita fugas de información del test hacia el entrenamiento.
# (California) train_test_split devuelve 4 trozos: X e y de train y de test.
# test_size=0.2 -> 20% para test, 80% para train. random_state=SEED garantiza
#   la misma partición en cada ejecución (reproducibilidad del resultado).
X_cal_train, X_cal_test, y_cal_train, y_cal_test = train_test_split(
    X_cal, y_cal, test_size=0.2, random_state=SEED)
print(f'California -> Train: {X_cal_train.shape[0]} | Test: {X_cal_test.shape[0]}')
# (Bank) Definimos aquí X_bnk y y_bnk: X = todas las columnas menos el target.
X_bnk = df_model.drop(columns=['Exited'])
y_bnk = df_model['Exited']
# concepto clave del challenge (STRATIFICACIÓN):
#   stratify=y_bnk respeta la proporción de clases (79.63/20.37) tanto en train como
#   en test. Sin esto, por puro azar el test podría quedarse sin “se van” -> mala evaluación.
X_bnk_train, X_bnk_test, y_bnk_train, y_bnk_test = train_test_split(
    X_bnk, y_bnk, test_size=0.2, random_state=SEED, stratify=y_bnk)
print(f'Bank     -> Train: {X_bnk_train.shape[0]} | Test: {X_bnk_test.shape[0]}')
# Verificamos que la proporción de bajas sea parecida en train y test (gracias a stratify).
print(f'Proporción de bajas en train: {y_bnk_train.mean():.4f} | test: {y_bnk_test.mean():.4f}')


## Escalado y pipelines

Estas transformaciones se encapsulan en **Pipelines** para evitar fuga de datos:

- **California**: `ColumnTransformer` que imputa `total_bedrooms` con la mediana, estandariza las numéricas y one-hot-codifica `ocean_proximity` (drop_first). La regresión lineal se anida en el pipeline para que el preprocesamiento se aprenda solo con `train` y no se filtre información del test.
- **Bank Churn**: `StandardScaler` para todas las columnas (ya numéricas tras el encoding), ajustado solo con train.

Así, el scaler se aprende únicamente con `X_train` y se aplica a test sin re-entrenar.


In [ ]:
# celda 14 - escalado y pipelines: california
# concepto clave del challenge (Paso 6 — Preprocesamiento + pipeline):
#   Un Pipeline encadena transformaciones + modelo en un solo objeto. La ventaja:
#   las transformaciones se aprenden SOLO con train y se aplican a test de forma
#   automática, imposibilitando la fuga de datos.
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
# ColumnTransformer: permite aplicar transformaciones DISTINTAS a grupos de columnas.
prepro_cal = ColumnTransformer([
    # Grupo 'num' (numéricas): imputamos nulos con la mediana (SimpleImputer median)
    #   y luego estandarizamos (StandardScaler -> media 0, desviación estándar 1).
    #   El escalado es clave para comparar coeficientes de variables con escalas distintas.
    ('num', Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('sc', StandardScaler())
    ]), num_cols + geo_cols),
    # Grupo 'cat' (categóricas): imputamos el valor más frecuente y aplico one-hot
    #   con drop_first (k-1 dummies) y handle_unknown='ignore' (no explota si hay clase nueva).
    ('cat', Pipeline([
        ('imp', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(drop='first', handle_unknown='ignore'))
    ]), cat_cols)
])


## Entrenamiento (las Misiones)

### Misión 1 — Regresión sobre California Housing

1. **Regresión Lineal clásica** (OLS).

### Misión 2 — Clasificación sobre Churn

**Regresión Logística** para estimar la probabilidad de abandono. Se comparan dos variantes: la base y una con `class_weight='balanced'` para compensar el desbalance de clases.


In [ ]:
# celda 16 - misión 1: regresion lineal (california)
# concepto del challenge: esta es la MISIÓN 1 del desafío.
#   Regresión Lineal (OLS) para predecir el precio.
from sklearn.preprocessing import OneHotEncoder
# misión 1: regresión lineal
# Pipeline: primero se preprocesan los datos (prepro_cal) y luego se entrena el modelo.
lin_pipe = Pipeline([
    ('prep', prepro_cal),
    ('model', LinearRegression())   # modelo de regresión lineal (mínimos cuadrados / OLS)
])
# .fit(): el modelo “aprende” los coeficientes que minimizan el error sobre train.
lin_pipe.fit(X_cal_train, y_cal_train)
print('Regresión Lineal entrenada.')


In [ ]:
# celda 17 - misión 2: regresion logística (bank churn)
# concepto del challenge: MISIÓN 2 del desafío.
#   Regresión Logística para CLASIFICAR si un cliente se va (1) o se queda (0).
#   Comparamos dos variantes: base y con class_weight='balanced' (para el desbalance).
# Estandarizamos las características de Bank directamente (X_bnk_train).
#   fit_transform aprende media/desv. con train; .transform SOLO aplica (sin reentrenar)
#   -> el test no “contamina” el escalado (evita fuga de datos).
scaler_bnk = StandardScaler()
X_bnk_train_sc = scaler_bnk.fit_transform(X_bnk_train)
X_bnk_test_sc = scaler_bnk.transform(X_bnk_test)
# Modelo base: regresión logística (función sigmoide que devuelve una probabilidad 0-1).
# max_iter=2000 garantiza convergencia; random_state=SEED reproduce el resultado.
logreg = LogisticRegression(max_iter=2000, random_state=SEED)
logreg.fit(X_bnk_train_sc, y_bnk_train)
# concepto del challenge: DESBALANCE. class_weight='balanced' penaliza más los errores
#   sobre la clase minoritaria (“se van”) al re-ponderar automáticamente las clases.
#   Esto debería mejorar el RECALL (capturar más fugas) a costa de algo de precision.
logreg_bal = LogisticRegression(max_iter=2000, random_state=SEED, class_weight='balanced')
logreg_bal.fit(X_bnk_train_sc, y_bnk_train)
print('Regresión Logística entrenada (base y balanceada).')


## Evaluación

**Misión 1 (regresión):** `RMSE` y `R²` sobre **test**.
**Misión 2 (clasificación):** `Accuracy`, `Precision` y `Recall` sobre **test**, comparando el modelo base con el balanceado.


In [ ]:
# celda 19 - evaluación misión 1: rmse y r² (california)
# concepto clave del challenge (Paso 7 — Evaluación de regresión):
#   Evaluamos SÓLO con datos que el modelo no vio (test). El RMSE es el error promedio
#   en dólares reales; R² cuánto de la variación explica el modelo.
print('===== EVALUACIÓN — RMSE (Calidad del precio predicho) =====')
# .predict(): el modelo predice sobre test.
pred_lin = lin_pipe.predict(X_cal_test)
# concepto: RMSE = raíz del error cuadrático medio.
#   Se eleva al cuadrado (para castigar errores grandes) y se saca raíz para volver
#   a las unidades originales (dólares). Idealmente < std(y) (predecir mejor que la media).
rmse_lin = np.sqrt(mean_squared_error(y_cal_test, pred_lin))
# R² = 1 - (varianza residual / varianza total). 0.63 => el modelo explica el 63%.
r2_lin = r2_score(y_cal_test, pred_lin)
print(f'RMSE Regresión Lineal : {rmse_lin:,.0f} $')
print(f'R² Regresión Lineal : {r2_lin:.4f}')
print(f'\nstd(y) = {np.std(y_cal):,.0f} $')


In [ ]:
# celda 20 - evaluación misión 2: accuracy / precision / recall
# concepto clave del challenge (Evaluación de clasificación):
#   El Accuracy por sí solo MIENTE con clases desbalanceadas. Por eso también medimos
#   Precision y Recall. Recordemos: positivo = 1 = “el cliente se va”.
print('===== EVALUACIÓN — BANK CHURN =====')
# Predicciones de ambos modelos sobre el test estandarizado.
pred_bnk = logreg.predict(X_bnk_test_sc)
pred_bnk_bal = logreg_bal.predict(X_bnk_test_sc)
# Métricas del modelo BASE:
acc = accuracy_score(y_bnk_test, pred_bnk)    # % de aciertos total.
prec = precision_score(y_bnk_test, pred_bnk)  # de los “Se va” predichos, cuántos eran reales.
rec = recall_score(y_bnk_test, pred_bnk)      # de los que se van, cuántos atrapó.
# Métricas del modelo BALANCEADO (iguales fórmulas).
acc_b = accuracy_score(y_bnk_test, pred_bnk_bal)
prec_b = precision_score(y_bnk_test, pred_bnk_bal)
rec_b = recall_score(y_bnk_test, pred_bnk_bal)
# Imprimimos una tabla comparativa.
print(f'{"Modelo":<14}{"Accuracy":<12}{"Precision":<12}{"Recall"}')
print(f'{"Base":<14}{acc:<12.4f}{prec:<12.4f}{rec:.4f}')
print(f'{"Balanceado":<12}{acc_b:<12.4f}{prec_b:<12.4f}{rec_b:.4f}')
# Matriz de confusión: cruza real (filas) vs predicho (columnas).
#   [0,0]=correctos “se queda”, [1,1]=correctos “se va”, [0,1]=falsos positivos, [1,0]=falsos negativos.
print('\nMatriz de confusión (base):')
cm = confusion_matrix(y_bnk_test, pred_bnk)
print(cm)
# classification_report: resumen completo (precision, recall, f1 por clase).
print('\nReporte de clasificación (base):')
print(classification_report(y_bnk_test, pred_bnk, target_names=['Se queda', 'Se va']))


## ¿Memoriza o generaliza? (train vs test)

*“Un modelo que memoriza todo el dataset no es brillante. Es tóxico.”*

La prueba de fuego contra el sobreajuste es comparar el error en **train** vs **test**: un modelo que memoriza brilla en train y colapsa en test. Aquí verificamos que aprendemos patrones, no que replicamos datos de memoria.


In [ ]:
# celda 21 - anti-memorización: train vs test
# concepto clave del challenge: “el modelo que memoriza el dataset no es brillante, es tóxico”.
#   Un modelo con overfitting tendría error ~0 en train y error enorme en test.
#   Si test ≈ train, el modelo GENERALIZA: aprendió patrones, no replicó guiones de memoria.
print('===== ¿MEMORIZA O GENERALIZA? (train vs test) =====')

# --- Regresión (California): RMSE en train vs test, SIN re-entrenar ---
print('\nREGRESIÓN (California) — RMSE:  train vs test')
for nombre, p_tr, p_te in [
    ('Linear', lin_pipe.predict(X_cal_train), pred_lin),
]:
    rmse_tr = np.sqrt(mean_squared_error(y_cal_train, p_tr))
    rmse_te = np.sqrt(mean_squared_error(y_cal_test,  p_te))
    dife = rmse_te - rmse_tr
    veredicto = 'generaliza (sin memorizar)' if abs(dife) / rmse_tr < 0.05 else 'REVISAR: posible sobreajuste'
    print(f'  {nombre:<7}: train={rmse_tr:>10,.0f} $ | test={rmse_te:>10,.0f} $ | Δ={dife:>+9,.0f} $  -> {veredicto}')

# --- Clasificación (Churn, Logística base): Accuracy y Recall en train vs test ---
acc_tr = accuracy_score(y_bnk_train, logreg.predict(X_bnk_train_sc))
rec_tr = recall_score(y_bnk_train, logreg.predict(X_bnk_train_sc))
print('\nCLASIFICACIÓN (Churn, Logística base) — train vs test')
print(f'  Accuracy:  train={acc_tr:.4f} | test={acc:.4f}')
print(f'  Recall:    train={rec_tr:.4f} | test={rec:.4f}')
print('  Conclusión: si test ≈ train, no está memorizando (y estamos del lado bueno del overfitting).')


## Diagnóstico visual

Los números resumen el desempeño; los gráficos revelan **cómo** falla el modelo: heterocedasticidad, subestimación de alto valor o dónde se concentran los errores de clasificación.


In [ ]:
# celda 22 - diagnóstico visual: regresion (residuos)
# concepto del challenge: Los números resumen, los gráficos revelan CÓMO falla el modelo.
#   Analizar los RESIDUOS (error por observación) detecta subestimaciones y heterocedasticidad.
# Residuo = valor real - predicción (cuánto se equivoca el modelo en cada casa).
residuos = y_cal_test - pred_lin
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Regresión Lineal — Diagnósticos', fontsize=14)
# (1) Predicción vs Real: si el modelo fuera perfecto, los puntos caerían sobre la línea roja.
axes[0].scatter(y_cal_test, pred_lin, alpha=0.3, s=10)
# Línea diagonal (y=x) referencia de predicción perfecta.
axes[0].plot([y_cal_test.min(), y_cal_test.max()], [y_cal_test.min(), y_cal_test.max()], 'r--', lw=2)
axes[0].set_xlabel('Valor real'); axes[0].set_ylabel('Predicción'); axes[0].set_title('Predicción vs Real')
# (2) Residuos vs Predicción: si se abren en abanico (“cono”), hay heterocedasticidad
#   -> el modelo subestima las casas caras (residuos negativos donde predice alto valor).
axes[1].scatter(pred_lin, residuos, alpha=0.3, s=10)
axes[1].axhline(0, color='r', linestyle='--', lw=2)  # línea en residuo=0 (error nulo)
axes[1].set_xlabel('Predicción'); axes[1].set_ylabel('Residuo'); axes[1].set_title('Residuos vs Predicción')
# (3) Histograma de residuos: si es simétrico alrededor de 0, el error no tiene sesgo.
axes[2].hist(residuos, bins=50, edgecolor='black'); axes[2].axvline(0, color='r', linestyle='--', lw=2)
axes[2].set_xlabel('Residuo'); axes[2].set_title('Distribución de Residuos')
plt.tight_layout(); plt.show()


In [ ]:
# celda 23 - diagnóstico visual: clasificación (matrices de confusión)
# concepto del challenge: la matriz de confusión muestra exactamente DÓNDE se equivoca
#   cada modelo. Al comparar base vs balanceado vemos el trade-off precision/recall.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# ConfusionMatrixDisplay grafica la matriz con colores para el modelo base.
ConfusionMatrixDisplay(confusion_matrix(y_bnk_test, pred_bnk),
                       display_labels=['Se queda', 'Se va']).plot(ax=axes[0], colorbar=False)
axes[0].set_title('Modelo base')
# Ídem para el modelo balanceado.
ConfusionMatrixDisplay(confusion_matrix(y_bnk_test, pred_bnk_bal),
                       display_labels=['Se queda', 'Se va']).plot(ax=axes[1], colorbar=False)
axes[1].set_title('Modelo balanceado')
plt.suptitle('Comparación de matrices de confusión', fontsize=13)
plt.tight_layout(); plt.show()
# concepto: Falsos positivos = clientes que NO se van pero el modelo marcó como “se van”.
#           Falsos negativos = clientes que SÍ se van pero el modelo NO detectó.
#   En retención, el FALSO NEGATIVO es lo que más importa (cliente que se pierde en silencio).
print('\nComparación de errores:')
print(f'  Base      : Falsos positivos={cm[0,1]}, Falsos negativos={cm[1,0]}')
cm_b = confusion_matrix(y_bnk_test, pred_bnk_bal)
print(f'  Balanceado: Falsos positivos={cm_b[0,1]}, Falsos negativos={cm_b[1,0]}')


## Interpretación de coeficientes

Se analiza el peso que el modelo le asignó a cada feature de California, en unidades comparables gracias al `StandardScaler` del pipeline: cada coeficiente expresa el cambio en dólares por cada desviación estándar extra de esa feature, manteniendo el resto constante.


In [ ]:
# celda 26 - interpretación de coeficientes (regresión de California)
# concepto clave del challenge: con features estandarizadas, el coeficiente dice
#   “cuántos dólares cambia el precio por cada desviación estándar extra de esta
#   feature, con el resto constante”. Es el POR QUÉ detrás del RMSE.
# El pipeline guarda todo: .named_steps['model'] es la regresión entrenada y
#   .named_steps['prep'] el preprocesador con los nombres reales de las columnas.
nombres_cal = lin_pipe.named_steps['prep'].get_feature_names_out()
coefs_cal = lin_pipe.named_steps['model'].coef_
tabla_coef = pd.DataFrame({'feature': nombres_cal, 'coeficiente': coefs_cal})
tabla_coef['feature'] = tabla_coef['feature'].str.replace(r'^(num|cat)__', '', regex=True)
tabla_coef['abs'] = tabla_coef['coeficiente'].abs()
tabla_coef = tabla_coef.sort_values('abs', ascending=False).drop(columns='abs')
print(f'Intercepto (B0): {lin_pipe.named_steps["model"].intercept_:,.0f} $')
print(tabla_coef.to_string(index=False))


### Interpretación (California)

- **median_income** es el predictor dominante (**+43,817** $ por desviación estándar) — coherente con la correlación alta observada en el EDA.
- **latitude** (**-50,765**) y **longitude** (**-40,170**) confirman que la ubicación cruda sigue pesando: más al norte o al interior (este del estado) el precio baja. Es la señal que las zonas geográficas de K-Means capturan de forma interpretable.
- **ocean_proximity** funciona como esperábamos: estar en **INLAND** reduce el precio (**-25,314**) y en **NEAR OCEAN** lo sube (**+23,941**).
- **ocean_proximity_ISLAND** tiene un coeficiente enorme (**+181,767**) pese a que representa menos del 1% del dataset: el modelo tiene poca evidencia para estimarlo con confianza. Es una señal de inestabilidad, no una regla de mercado.
- **population** (**-40,986**) y **households** (**+37,996**) muestran las dos caras de la densidad urbana: más población por bloque barata el precio; más hogares (vivienda familiar) lo sostiene.
- **rooms_per_household** (**+4,109**) y **population_per_room** (**+396**) aportan poco una vez que `median_income`, la ubicación y la interacción ya están en el modelo: su señal ya quedó absorbida por variables más fuertes.
- **age_x_income** (**+13,193**) suma después del ingreso: las casas viejas en barrios ricos valen más que la suma de “vieja” + “rica” por separado (interacción real capturada).


## Interpretación humana

### ¿Está el RMSE más cerca de $5 o de $500,000?

El target está en dólares reales. Tras eliminar el techo de $500,001, los precios van de ~$15K a ~$500K. Un RMSE de ~$60K significa que el modelo se equivoca, en promedio, en unos **$60,000** por vivienda. Eso está **mucho más cerca de $500,000 que de $5**.

¿Es un fracaso? **No.** Es la naturaleza del problema: con variables tabulares y un mercado con dinámicas no lineales, predecir el precio a $5 es imposible. La pregunta correcta es si el modelo supera la línea base de adivinar la media constante — y lo hace.


In [ ]:
# celda 30 - interpretación humana del rmse (¿más cerca de $5 o de $500k?)
# concepto clave del challenge: el desafío pregunta si el RMSE está más cerca de $5
#   (predicción de genio) o de $500K (predicción mala). Debemos CONECTUALIZAR la escala.
# Error relativo: RMSE / std(y). Si <1, el modelo es mejor que simplemente adivinar la media.
error_rel = rmse_lin / max(np.std(y_cal), 1e-9)
# Rango real de precios en test (de ~15K a ~500K).
print(f'Rango del target (test): [{y_cal_test.min():,.0f} $, {y_cal_test.max():,.0f} $]')
# RMSE en dólares: el error promedio del modelo por casa.
print(f'RMSE (Regresión Lineal): {rmse_lin:,.0f} $')
# Respuesta del challenge: ~$60K está mucho más cerca de $500K que de $5.
print(f'¿Más cerca de $5 o de $500K? → Definitivamente MÁS CERCA de $500K.')
# Nota: <1 => mejor que la media constante. ~0.62 => explica buena parte de la volatilidad.
print(f'RMSE relativo a la volatilidad del precio (std): {error_rel:.2f} (< 1 = mejor que adivinar la media)')


### Precision, Recall y Accuracy en el contexto del *cliente que se va*

Definimos la clase **positiva = 1 = “el cliente abandona el banco”**.

- **Accuracy** = fracción de aciertos sobre todas las predicciones. **Puede mentir** con clases desbalanceadas: un modelo que siempre dice “se queda” obtendría ~79.6% de accuracy sin atrapar a nadie.
- **Precision** = de los clientes marcados como fugables, ¿cuántos lo eran realmente? Alta precision → cuando el modelo dice “ése se va”, hazle caso (evita falsas alarmas).
- **Recall** = de los que realmente se van, ¿cuántos logró atrapar? Bajo recall → clientes a punto de irse pasan desapercibidos y se pierden en silencio (falso negativo = **cliente perdido**).

En retención suele priorizarse **recall**: es más barato falsamente pensar que alguien se va (y hacerle una oferta) que perderlo para siempre. La comparación base vs balanceado muestra exactamente ese trade-off.


In [ ]:
# celda 32 - reflexión sobre precision vs recall (churn)
# concepto clave del challenge: entendemos por qué el Accuracy engaña aquí y por qué
#   la elección del modelo depende del COSTO de cada error (no de un solo número).
print('REFLEXIÓN SOBRE PRECISION vs RECALL EN BANK CHURN:\n')
# Recall base ~0.19 => solo atrapa al 19% de los que realmente se iban.
print(f'- Recall (base) = {rec:.2f} → detecta al {rec*100:.0f}% de los que realmente se iban;')
# El 81% restante se va sin ser detectado (falso negativo = cliente perdido).
print(f'  el {(1-rec)*100:.0f}% restante se marcha sin ser señalado (falso negativo = CLIENTE PERDIDO).')
# Precision base ~0.59 => de todos los marcados como “se van”, 59% lo eran (41% falsa alarma).
print(f'- Precision (base) = {prec:.2f} → del {(pred_bnk==1).sum()} marcados como fugables,')
print(f'  solo el {prec*100:.0f}% lo eran; el resto es falsa alarma.')
# Accuracy base ~0.81 => alto, pero oculta el bajo recall.
print(f'- Accuracy (base) = {acc:.2f} → el {acc*100:.0f}% de TODAS las predicciones fueron correctas.')
# El balanceado sube recall (0.70) pero baja precisión (0.39).
print(f'\n- Balanceado: Recall={rec_b:.2f} (más fugables atrapados) a costa de Precision={prec_b:.2f}.')
print('  La elección del modelo depende del costo de cada error, no de un único número.')


## Conclusiones

### Desempeño del modelo — Regresión (California Housing)

| Modelo | RMSE | R² |
|--------|------|------|
| Regresión Lineal | $60,883 | 0.6282 |

El modelo explica el **62.8%** de la variación en los precios de vivienda, con un error promedio de aproximadamente **$60,883** por vivienda (tras eliminar el techo artificial de $500,001). El error restante convive con el rango real del mercado ($15K–$500K): es un error relativo razonable para una regresión lineal tabular.

### Desempeño del modelo — Clasificación (Bank Churn)

| Metrica | Modelo base | Modelo balanceado |
|---------|-------------|-------------------|
| Accuracy | 0.8080 | 0.7135 |
| Precision | 0.5891 | 0.3872 |
| Recall | 0.1867 | 0.7002 |

El modelo base supera en Accuracy (0.81) al baseline de 79.63% (un modelo que siempre predijera “Se queda”), pero con un **Recall de apenas 0.19**: atrapa sólo al 19% de los clientes que realmente se van. El desbalance hace que Accuracy por sí sola sea profundamente engañosa. El modelo balanceado sacrifica Accuracy (0.71) para subir Recall a 0.70, atrapando al 70% de los que se van.

### Qué funcionó

- **`median_income`** confirmó ser el predictor dominante del precio, consistente desde el EDA hasta los coeficientes finales (+$43,817 por desviación estándar).
- **Feature engineering**: los ratios (`rooms_per_household`, `population_per_room`) y las **zonas geográficas con K-Means** aportaron señal e interpretabilidad frente a las coordenadas crudas; `age_x_income` capturó una interacción invisible por separado.
- En bank churn, **`Age`** es el predictor más fuerte (+0.74): los clientes mayores abandonan más (edad media 44.8 vs 37.4). Le siguen **`Geography_Germany`** (+0.36, Alemania concentra el 32% de bajas) y el efecto protector de **`IsActiveMember`** (-0.52).

### La decisión que importa: ¿qué modelo de churn usar?

El modelo base tiene Precision alta (0.59) pero Recall muy bajo (0.19): casi nadie que se va es detectado. El balanceado tiene el patrón inverso. Ambos comparten F1 bajo (~0.28 base, ~0.50 balanceado), así que la elección depende del **costo de cada error** para el negocio:

- **Modelo base** → mayor Precision (0.59). Útil cuando los recursos de retención son limitados y hay que ser selectivo con las alertas (pocas, pero más certeras).
- **Modelo balanceado** → mayor Recall (0.70). Útil cuando perder un cliente es más caro que una falsa alarma. En banca minorista con alta competencia, este suele ser el escenario real (evitar la fuga de un cliente rentable importa más que el costo de una oferta puntual).

### Limitaciones detectadas

- El diagnóstico de residuos muestra que el modelo **subestima sistemáticamente las casas de alto valor** (heterocedasticidad visible en residuos vs predicción): el mercado de lujo responde a dinámicas no lineales que una regresión lineal no captura.
- El coeficiente de `ocean_proximity_ISLAND` es enorme (+$181,767) pese a que representa menos del 1% de los datos: poca evidencia para estimarlo con confianza. Es una señal de inestabilidad, no una regla de mercado.
- Con datos desbalanceados, el **Accuracy por sí solo es insuficiente**: en bank churn un modelo con 81% de accuracy dejaba escapar al 81% de los que se iban. Precision/Recall y la matriz de confusión son imprescindibles para elegir modelo.

### Errores que cometí y cómo el RMSE me humilló

- **Error 1 — El techo de $500,001**: al principio modelé con el target recortado. El RMSE era altísimo por culpa de esas casas de lujo “recortadas”. Al eliminar el techo (decisión del EDA), el error bajó y el modelo quedó más realista. **Me humilló** porque el problema no era mi modelo: eran mis datos.
- **Error 2 — Leer el RMSE sin la escala del negocio**: por un instante pensé que $60K era “muchísimo”. Hasta que miré el rango real ($15K–$500K): el RMSE no miente, yo no contextualizaba las unidades. Un error de $60K sobre casas de $500K es un error relativo razonable; $5 sería predicción perfecta, imposible con datos tabulares.
- **Error 3 — Confiar en Accuracy sin ver el desbalance**: en bank churn el modelo base presumía 81% de accuracy, pero su recall era de apenas 0.19: no estaba atrapando al 81% de los clientes que se iban. Solo al leer precision/recall y la matriz de confusión entendí que el modelo perfecto en % era débil para el negocio.
- **Error 4 — No verificar si memorizo o generalizo**: tras entrenar, comparé el RMSE en **train vs test** para detectar sobreajuste: la diferencia fue < 5%, señal de que el modelo **generaliza** y no se limitó a memorizar las respuestas del entrenamiento (el modelo que memoriza no es brillante, es tóxico).

### Qué haría distinto

- Verificar rangos y distribución (`.describe()`) de cada feature nueva inmediatamente después de crearla, antes de integrarla al pipeline.
- Investigar **VIF** y errores estándar de los coeficientes para respaldar formalmente las decisiones de multicolinealidad.
- Explorar el **ajuste del umbral de decisión** (<0.5) en churn para priorizar recall sin destruir precision, y probar **SMOTE** como alternativa a `class_weight='balanced'`.

---

**El RMSE me enseñó humildad.** No busco predicción perfecta; busco evaluación honesta, decisiones consistentes y métricas contextualizadas. El modelo es el instrumento; el humano decide la política.


In [ ]:
# celda 34 - cierre del ritual de los 7 pasos
# concepto del challenge: recordamos que completamos las dos misiones del desafío:
#   Misión 1 (regresión -> RMSE) y Misión 2 (clasificación -> Accuracy/Precision/Recall),
#   con interpretación y las conclusiones honestas/documentadas.
print('✓ Ritual completado: regresión (RMSE), clasificación (Accuracy/Precision/Recall), interpretación y conclusiones honestas.')
